In [1]:
print("ok")

ok


In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()

    return documents

In [ ]:
extracted_data = load_pdf_file(data='../Data/')


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNADH+Berkeley-BookItalic', '/FirstChar': 32, '/LastChar': 111, '/Subtype': '/Type1', '/FontDescriptor': IndirectObject(8788, 0, 3276474685088), '/Widths': [241, 241, 352, 481, 481, 667, 778, 204, 315, 315, 500, 481, 241, 333, 241, 278, 481, 481, 481, 481, 481, 481, 481, 481, 481, 481, 241, 241, 481, 481, 481, 352, 800, 593, 519, 611, 685, 537, 519, 667, 685, 296, 296, 611, 500, 796, 685, 685, 519, 667, 556, 463, 556, 685, 593, 870, 593, 519, 556, 315, 278, 315, 481, 500, 241, 463, 444, 370, 463, 389, 259, 407, 463, 241, 241, 444, 241, 722, 481, 426]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNAFK+Berkeley-Bold', '/FirstChar': 32,

In [15]:
len(extracted_data)

637

In [10]:
import os
print(os.getcwd())

c:\Users\deepe\Desktop\jai ganesh jai hoo bharu maharaj please ani bar chalad do\research


In [16]:
total_documents = len(extracted_data)
print(f"Total documents: {total_documents}")

Total documents: 637


In [17]:
# Split the Data into Text Chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [18]:
text_chunks = text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5860


In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

In [21]:
# Download the Embeddings from Hugging Face

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [22]:
embeddings = download_hugging_face_embeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [23]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [24]:
query_result

[-0.03447723388671875,
 0.031023165211081505,
 0.006734973751008511,
 0.026108991354703903,
 -0.039362046867609024,
 -0.16030244529247284,
 0.06692394614219666,
 -0.0064415112137794495,
 -0.04745052382349968,
 0.014758817851543427,
 0.07087530195713043,
 0.05552754923701286,
 0.019193369895219803,
 -0.026251282542943954,
 -0.01010951679199934,
 -0.02694047801196575,
 0.022307446226477623,
 -0.022226616740226746,
 -0.1496925950050354,
 -0.017493078485131264,
 0.007676276378333569,
 0.054352231323719025,
 0.0032544422429054976,
 0.03172587975859642,
 -0.0846213772892952,
 -0.029405953362584114,
 0.05159562826156616,
 0.04812408611178398,
 -0.0033148485235869884,
 -0.05827920883893967,
 0.04196930304169655,
 0.022210706025362015,
 0.128188818693161,
 -0.02233896777033806,
 -0.011656241491436958,
 0.06292831897735596,
 -0.032876331359148026,
 -0.09122606366872787,
 -0.031175393611192703,
 0.05269952118396759,
 0.047034814953804016,
 -0.0842030942440033,
 -0.030056195333600044,
 -0.02074482

In [25]:
from dotenv import load_dotenv
load_dotenv()

True

In [26]:
PINCONE_API_KEY = os.environ.get('PINCONE_API_KEY')

In [28]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=PINCONE_API_KEY)

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-x5x47jb.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [30]:
index = pc.Index("medicalbot")

print("Connected to Pinecone index successfully")

Connected to Pinecone index successfully


In [35]:
# Embed each chunk and upload the embeddings to Pinecone

vectors = []

for i, chunk in enumerate(text_chunks):
    vector = embeddings.embed_query(chunk.page_content)

    vectors.append({
        "id": f"doc-{i}",
        "values": vector,
        "metadata": {
            "text": chunk.page_content,
            "page": chunk.metadata.get("page"),
            "source": chunk.metadata.get("source")
            
}
    })

# Upload vectors in batches
batch_size = 100

for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i:i + batch_size])

print(f"Uploaded {len(vectors)} vectors to Pinecone")

Uploaded 5860 vectors to Pinecone


In [71]:
from dotenv import load_dotenv
import os
from huggingface_hub import InferenceClient

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

client = InferenceClient(
    api_key=HF_TOKEN,
    provider="featherless-ai"
)

print("Featherless AI provider connected")

Featherless AI provider connected


In [36]:
print("Total text chunks:", len(text_chunks))
print("Vectors created:", len(vectors))

Total text chunks: 5860
Vectors created: 5860


In [ ]:
# Load the existing Pinecone index

index = pc.Index("medicalbot")

print("Existing  Pinecone index loaded successfully")

Existing Pinecone index loaded successfully


In [38]:
docsearch = index

docsearch

In [42]:
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(
    lambda query: index.query(
        vector=embeddings.embed_query(query),
        top_k=3,
        include_metadata=True
    )
)

In [43]:
retrieved_docs = retriever.invoke("What is Acne?")

In [44]:
print(retrieved_docs)

{'matches': [{'id': 'doc-299',
              'metadata': {'page': 39.0,
                           'source': '..\\Data\\Medical_book.pdf',
                           'text': 'GALE ENCYCLOPEDIA OF MEDICINE 226\n'
                                   'Acne\n'
                                   'GEM - 0001 to 0432 - A  10/22/03 1:41 PM  '
                                   'Page 26'},
              'score': 0.745599866,
              'values': []},
             {'id': 'doc-289',
              'metadata': {'page': 38.0,
                           'source': '..\\Data\\Medical_book.pdf',
                           'text': 'GALE ENCYCLOPEDIA OF MEDICINE 2 25\n'
                                   'Acne\n'
                                   'Acne vulgaris affecting a woman’s face. '
                                   'Acne is the general\n'
                                   'name given to a skin disorder in which the '
                                   'sebaceous\n'
                            

In [45]:
retrieved_docs

{'matches': [{'id': 'doc-299',
              'metadata': {'page': 39.0,
                           'source': '..\\Data\\Medical_book.pdf',
                           'text': 'GALE ENCYCLOPEDIA OF MEDICINE 226\n'
                                   'Acne\n'
                                   'GEM - 0001 to 0432 - A  10/22/03 1:41 PM  '
                                   'Page 26'},
              'score': 0.745599866,
              'values': []},
             {'id': 'doc-289',
              'metadata': {'page': 38.0,
                           'source': '..\\Data\\Medical_book.pdf',
                           'text': 'GALE ENCYCLOPEDIA OF MEDICINE 2 25\n'
                                   'Acne\n'
                                   'Acne vulgaris affecting a woman’s face. '
                                   'Acne is the general\n'
                                   'name given to a skin disorder in which the '
                                   'sebaceous\n'
                            

In [47]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

In [50]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [75]:
from langchain_core.runnables import RunnableLambda

def format_docs(result):
    return "\n\n".join(
        match["metadata"]["text"]
        for match in result["matches"]
    )

rag_chain = (
    {
        "context": RunnableLambda(
            lambda x: retriever.invoke(x["input"])
        ) | format_docs,
        "input": RunnableLambda(lambda x: x["input"])
    }
    | prompt
    | llm
    | RunnableLambda(lambda x: {"answer": x})
)

print("RAG chain created successfully")

RAG chain created successfully


In [73]:
from langchain_core.runnables import RunnableLambda

def hf_generate(prompt_value):
    messages = []

    for msg in prompt_value.to_messages():
        role = "system" if msg.type == "system" else "user"
        messages.append({
            "role": role,
            "content": msg.content
        })

    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=messages,
        max_tokens=200,
        temperature=0.4
    )

    return response.choices[0].message.content

llm = RunnableLambda(hf_generate)

print("Hugging Face LLM ready")

Hugging Face LLM ready


In [65]:
print("Token exists:", bool(HF_TOKEN))
print("Token starts with hf_:", HF_TOKEN.startswith("hf_"))

Token exists: True
Token starts with hf_: True


In [69]:
from huggingface_hub import model_info

info = model_info(
    "Qwen/Qwen2.5-1.5B-Instruct",
    expand="inferenceProviderMapping"
)

print(info.inference_provider_mapping)

[InferenceProviderMapping(provider='featherless-ai', hf_model_id='Qwen/Qwen2.5-1.5B-Instruct', provider_id='Qwen/Qwen2.5-1.5B-Instruct', status='live', task='conversational', adapter=None, adapter_weights_path=None, type=None)]


In [72]:
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is Acne?"
        }
    ],
    max_tokens=100
)

print(response.choices[0].message.content)

Acne, also known as acne vulgaris, is a common skin condition that affects the sebaceous glands (oil-producing glands) of the skin. These glands are responsible for producing an oily substance known as sebum, which helps to keep the skin moisturized. However, when the sebaceous glands become plugged with dead skin cells and leftover sebum, acne can develop.

Here are some key points about acne:

1. Causes:
- Hormonal changes
- Prolonged stress


In [74]:
response = rag_chain.invoke("What is Acne?")
print(response)

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria.


In [76]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

I don't know.


In [77]:
response = rag_chain.invoke({"input": "What is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder in which the abnormal release of a particular chemical from the pituitary gland in the brain causes increased growth in bone and soft tissue, as well as a variety of other disturbances throughout the body. This chemical released from the pituitary gland is known as growth hormone. Gigantism is a condition where excessive growth occurs before or after the normal growth period, leading to unusually tall stature.
